# Explore Teller sandbox data

Goal: see the shape of accounts, transactions, balances before writing the ingest script.

Run cells top-to-bottom. Each cell is small so we can pause and discuss.

In [1]:
import os
import json
import requests
from collections import Counter
from decimal import Decimal
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join('..', '.env'))

CERT  = os.path.join('..', os.environ['TELLER_CERT_PATH'])
KEY   = os.path.join('..', os.environ['TELLER_KEY_PATH'])
TOKEN = os.environ['TELLER_SANDBOX_ACCESS_TOKEN']
BASE  = 'https://api.teller.io'

def teller_get(path):
    """Helper: GET against Teller with mTLS cert + token auth."""
    r = requests.get(f'{BASE}{path}', cert=(CERT, KEY), auth=(TOKEN, ''))
    r.raise_for_status()
    return r.json()

print('cert ok:', os.path.exists(CERT))
print('key ok :', os.path.exists(KEY))
print('token  :', TOKEN[:15] + '...')

cert ok: True
key ok : True
token  : test_token_f4sj...


## 1. Accounts

What does an account look like? What types/subtypes show up?

In [2]:
accounts = teller_get('/accounts')
print(f'{len(accounts)} accounts\n')
print(json.dumps(accounts[0], indent=2))

3 accounts

{
  "last_four": "8946",
  "subtype": "credit_card",
  "institution": {
    "name": "Chase",
    "id": "chase"
  },
  "enrollment_id": "enr_prj2vvsum2jv9e8ik4000",
  "currency": "USD",
  "type": "credit",
  "status": "open",
  "name": "Platinum Card",
  "links": {
    "balances": "https://api.teller.io/accounts/acc_prj2vvsmjcjfj8s50e000/balances",
    "transactions": "https://api.teller.io/accounts/acc_prj2vvsmjcjfj8s50e000/transactions",
    "details": "https://api.teller.io/accounts/acc_prj2vvsmjcjfj8s50e000/details",
    "self": "https://api.teller.io/accounts/acc_prj2vvsmjcjfj8s50e000"
  },
  "id": "acc_prj2vvsmjcjfj8s50e000"
}


In [3]:
# Quick summary of all accounts
for a in accounts:
    print(f"  {a['type']:>10} / {a['subtype']:>12}  |  {a['institution']['name']:>10}  |  {a['name']:<25}  |  ****{a['last_four']}")

      credit /  credit_card  |       Chase  |  Platinum Card              |  ****8946
  depository /     checking  |       Chase  |  My Checking                |  ****9620
  depository /      savings  |       Chase  |  Essential Savings          |  ****1762


## 2. Balances

Balance is a separate endpoint per account. What does it return?

In [4]:
for a in accounts:
    bal = teller_get(f"/accounts/{a['id']}/balances")
    print(f"{a['name']:<25}  ledger={bal.get('ledger'):>12}  available={bal.get('available'):>12}")

# Show one full balance response to see all fields
print('\n--- full balance response ---')
print(json.dumps(bal, indent=2))

Platinum Card              ledger=     5945.80  available=     8988.63
My Checking                ledger=    86638.96  available=    86580.10
Essential Savings          ledger=    74381.26  available=    74266.71

--- full balance response ---
{
  "ledger": "74381.26",
  "account_id": "acc_prj2vvsljcjfj8s50e000",
  "available": "74266.71",
  "links": {
    "account": "https://api.teller.io/accounts/acc_prj2vvsljcjfj8s50e000",
    "self": "https://api.teller.io/accounts/acc_prj2vvsljcjfj8s50e000/balances"
  }
}


## 3. Transactions

Pull all transactions across all accounts. Look at one in detail, then aggregate.

In [5]:
all_txns = []
for a in accounts:
    txns = teller_get(f"/accounts/{a['id']}/transactions")
    all_txns.extend(txns)

print(f'{len(all_txns)} transactions total')
print('\n--- sample transaction ---')
print(json.dumps(all_txns[0], indent=2))

321 transactions total

--- sample transaction ---
{
  "running_balance": null,
  "account_id": "acc_prj2vvsmjcjfj8s50e000",
  "amount": "65.57",
  "details": {
    "processing_status": "complete",
    "counterparty": {
      "type": "organization",
      "name": "BIG BANK"
    },
    "category": "general"
  },
  "description": "International Transaction Fee",
  "date": "2026-04-27",
  "type": "fee",
  "status": "pending",
  "links": {
    "account": "https://api.teller.io/accounts/acc_prj2vvsmjcjfj8s50e000",
    "self": "https://api.teller.io/accounts/acc_prj2vvsmjcjfj8s50e000/transactions/txn_prj321g23cjfj8s50e000"
  },
  "id": "txn_prj321g23cjfj8s50e000"
}


In [6]:
# Status breakdown — how many pending vs posted?
print('--- status ---')
for k, v in Counter(t['status'] for t in all_txns).items():
    print(f'  {v:4d}  {k}')

print('\n--- categories ---')
for k, v in Counter(t['details']['category'] for t in all_txns).most_common():
    print(f'  {v:4d}  {k}')

print('\n--- transaction types ---')
for k, v in Counter(t['type'] for t in all_txns).most_common():
    print(f'  {v:4d}  {k}')

--- status ---
     3  pending
   318  posted

--- categories ---
    77  service
    61  general
    49  dining
    47  shopping
    14  groceries
    10  phone
    10  transportation
     9  accommodation
     9  entertainment
     8  software
     8  office
     7  health
     4  utilities
     3  electronics
     3  home
     2  fuel

--- transaction types ---
   134  card_payment
    27  atm
    26  payment
    24  ach
    20  withdrawal
    17  fee
    17  digital_payment
    14  deposit
    12  interest
     8  wire
     7  credit
     7  bill_payment
     3  check
     3  transfer
     2  adjustment


In [7]:
# Sign convention sanity check: positive = charge (you spent),
# negative = refund / income (money came in).
# We're storing as-is (no flip), so confirm what we'd be storing.

amounts = [Decimal(t['amount']) for t in all_txns]
pos = [a for a in amounts if a > 0]
neg = [a for a in amounts if a < 0]
zero = [a for a in amounts if a == 0]

print(f'positive (charges): {len(pos):4d}  sum={sum(pos):>12}')
print(f'negative (refunds): {len(neg):4d}  sum={sum(neg):>12}')
print(f'zero              : {len(zero):4d}')
print(f'\namount range: {min(amounts)} ... {max(amounts)}')

positive (charges):  131  sum=     8837.51
negative (refunds):  190  sum=   -12720.33
zero              :    0

amount range: -124.85 ... 124.95


In [8]:
# Counterparty: how often is it populated? Any nulls?
with_cp    = [t for t in all_txns if t['details'].get('counterparty', {}).get('name')]
without_cp = [t for t in all_txns if not t['details'].get('counterparty', {}).get('name')]

print(f'with counterparty   : {len(with_cp)}')
print(f'without counterparty: {len(without_cp)}')

if without_cp:
    print('\nExamples without counterparty:')
    for t in without_cp[:3]:
        print(f"  {t['date']}  {t['type']:<15}  {t['amount']:>10}  {t['description']}")

with counterparty   : 321
without counterparty: 0


In [9]:
# Date range of available data
dates = sorted(t['date'] for t in all_txns)
print(f'Earliest: {dates[0]}')
print(f'Latest  : {dates[-1]}')
print(f'Span    : {len(set(dates))} distinct days')

Earliest: 2026-01-28
Latest  : 2026-04-27
Span    : 89 distinct days


## 4. Edge cases to think about before ingest

After running the cells above, things worth discussing:

1. Are there transactions where `details.category` is missing or weird?
2. How do we want to handle the `service` and other unmapped categories?
3. Does Teller paginate transactions? (Sandbox returns everything in one call; real banks may not.)
4. What's the sign convention for `transfer` / `payment` / `deposit` types? Does it differ from `card_payment`?

In [10]:
# Sign by transaction type — sanity check that the sign matches what we'd expect
from collections import defaultdict
by_type = defaultdict(list)
for t in all_txns:
    by_type[t['type']].append(Decimal(t['amount']))

print(f"{'type':<18} {'n':>4} {'pos':>4} {'neg':>4} {'sum':>14}")
for ty, vals in sorted(by_type.items(), key=lambda x: -len(x[1])):
    pos = sum(1 for v in vals if v > 0)
    neg = sum(1 for v in vals if v < 0)
    print(f'{ty:<18} {len(vals):>4} {pos:>4} {neg:>4} {sum(vals):>14}')

type                  n  pos  neg            sum
card_payment        134   83   51        2438.61
atm                  27    9   18        -837.30
payment              26    2   24       -1241.17
ach                  24    4   20       -1151.47
withdrawal           20    0   20       -1435.12
fee                  17    9    8         127.33
digital_payment      17    0   17       -1200.50
deposit              14   14    0         974.99
interest             12    7    5          62.42
wire                  8    0    8        -537.37
credit                7    0    7        -413.45
bill_payment          7    0    7        -458.52
check                 3    3    0         142.27
transfer              3    0    3        -215.17
adjustment            2    0    2        -138.37
